###schema evolution


merge schema

In [0]:
%python
my_data=[(1,'food',10),(2,'drink',20),(3,'food',30),(4,'drink',40)]
my_schema= "id int, category string, amt int"
df = spark.createDataFrame(my_data, my_schema)
display(df)

id,category,amt
1,food,10
2,drink,20
3,food,30
4,drink,40


In [0]:
df_new = df.union(spark.createDataFrame([(5,'food',50)], schema=my_schema))

display(df_new)

id,category,amt
1,food,10
2,drink,20
3,food,30
4,drink,40
5,food,50


In [0]:
from pyspark.sql.functions import *
df_new=df_new.withColumn("flag",lit(1))

In [0]:
display(df_new)

id,category,amt,flag
1,food,10,1
2,drink,20,1
3,food,30,1
4,drink,40,1
5,food,50,1


In [0]:
%python
df_new.write.format('delta')\
  .mode('append')\
  .option("path","abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/sch_tbl")\
  .option("mergeSchema","true")\
  .save()

In [0]:
df=spark.read.format('delta').load("abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/sch_tbl")
display(df)

id,category,amt,flag
1,food,10,1
2,drink,20,1
3,food,30,1
4,drink,40,1
5,food,50,1
1,food,10,1
2,drink,20,1
3,food,30,1
4,drink,40,1
5,food,50,1


### explicit schema updates

add a coloumn

In [0]:
%sql
alter table delta_catalog.raw.ext_table_dml
add column flag string

In [0]:
%sql
select * from delta_catalog.raw.ext_table_dml


id,order_name,amount,prod_id,flag
2,order2,2002.0,200,null
3,order3,3003.0,300,null
1,order1,1001.0,100,null
2,order2,2002.0,200,null
3,order3,3003.0,300,null
1,order1,1001.0,100,null
2,order2,2002.0,200,null
3,order3,3003.0,300,null
1,order1,1001.0,100,null
2,order2,2002.0,200,null


## add a column after

In [0]:
%sql
alter table delta_catalog.raw.ext_table_dml
add column newcol string after id

In [0]:
%sql
select * from delta_catalog.raw.ext_table_dml

id,newcol,order_name,amount,prod_id,flag
2,null,order2,2002.0,200,null
3,null,order3,3003.0,300,null
1,null,order1,1001.0,100,null
2,null,order2,2002.0,200,null
3,null,order3,3003.0,300,null
1,null,order1,1001.0,100,null
2,null,order2,2002.0,200,null
3,null,order3,3003.0,300,null
1,null,order1,1001.0,100,null
2,null,order2,2002.0,200,null


### reordering columns

In [0]:
%sql
alter table delta_catalog.raw.ext_table_dml
alter column newcol after flag

In [0]:
%sql select * from delta_catalog.raw.ext_table_dml

id,order_name,amount,prod_id,flag,newcol
2,order2,2002.0,200,null,null
3,order3,3003.0,300,null,null
1,order1,1001.0,100,null,null
2,order2,2002.0,200,null,null
3,order3,3003.0,300,null,null
1,order1,1001.0,100,null,null
2,order2,2002.0,200,null,null
3,order3,3003.0,300,null,null
1,order1,1001.0,100,null,null
2,order2,2002.0,200,null,null


### rename column

In [0]:
%sql
ALTER TABLE delta_catalog.raw.ext_table_dml SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

In [0]:
%sql
alter table delta_catalog.raw.ext_table_dml
rename column newcol to newflag

In [0]:
%sql select * from delta_catalog.raw.ext_table_dml

id,order_name,amount,prod_id,flag,newflag
2,order2,2002.0,200,null,null
3,order3,3003.0,300,null,null
1,order1,1001.0,100,null,null
2,order2,2002.0,200,null,null
3,order3,3003.0,300,null,null
1,order1,1001.0,100,null,null
2,order2,2002.0,200,null,null
3,order3,3003.0,300,null,null
1,order1,1001.0,100,null,null
2,order2,2002.0,200,null,null


### reorg command

In [0]:
%sql
reorg table delta_catalog.raw.ext_table_dml apply (purge)

path,metrics
abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/ext_table_dml,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, true, 0, 0, 1776778231935, 1776778235137, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null)"
